# QLoRA + SFT 双卡 DDP 进阶版

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/qlora-sft-tutorial/blob/main/qlora_sft_ddp_tutorial.ipynb)

> **Kaggle 一键运行**：点击上方按钮 → 选择加速器 **"GPU T4 x2"**（注意必须是 x2 双卡）→ Run All


承接 [`qlora_sft_tutorial.ipynb`](./qlora_sft_tutorial.ipynb)（单卡 GPT-2 124M），本 notebook 演示如何把 QLoRA 训练扩展到 **双卡 DDP**：

| 维度 | 单卡版（上一个 notebook） | 双卡 DDP 版（本 notebook） |
|------|--------------------------|--------------------------|
| 模型 | GPT-2 124M | **Qwen2.5-1.5B**（12× 大） |
| 数据 | 8 条玩具样本 | **Alpaca 1000 条**真实指令数据 |
| GPU | 1 张 T4 | **2 张 T4 DDP**（吞吐 ~2×） |
| 启动方式 | 直接运行 | `accelerate.notebook_launcher` |
| Kaggle 运行时 | < 1 分钟 | ~10-15 分钟 |

## 学习目标
- 理解 **数据并行（DDP）** 与 **模型并行（device_map='auto'）** 的区别
- 掌握在 Jupyter notebook 内启动多进程分布式训练的标准做法
- 学会处理 DDP 下的常见陷阱（rank 0 才保存、find_unused_parameters 等）

## 前置要求
- 在 Kaggle 上请选择加速器 **"GPU T4 x2"**（不是 "GPU T4"）
- 建议先看完单卡版的 notebook


## Step 0：安装依赖

In [ ]:
# Kaggle / Colab 一键安装
import subprocess, sys

pkgs = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "datasets>=2.18.0",
    "accelerate>=0.29.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("依赖安装完毕")


## Step 1：检查双卡环境

> ⚠️ **关键约束**：在调用 `notebook_launcher` 之前，主 notebook 进程**绝对不能初始化 CUDA**。
> 因为 `notebook_launcher` 默认用 `fork` 启动子进程，fork 出来的子进程会继承父进程的 CUDA 上下文，
> 但又不能在子进程里重新初始化 → 报 `Cannot re-initialize CUDA in forked subprocess`。
>
> 这意味着：**前面任何 cell 都不能调用 `torch.cuda.*`**（如 `torch.cuda.is_available()`、
> `torch.cuda.get_device_properties()`、`.to('cuda')` 等）。所以下面用 `nvidia-smi` 子进程
> 检测 GPU 数量，绕开 torch CUDA 初始化。

In [ ]:
# 不能调用 torch.cuda.*！否则后面 notebook_launcher 用 fork 启子进程时会炸。
# 这里走 nvidia-smi 子进程查 GPU 信息，完全不碰 PyTorch 的 CUDA 上下文。
import subprocess, sys

try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    gpu_lines = out.split("\n") if out else []
    n_gpu = len(gpu_lines)
    print(f"检测到 {n_gpu} 张 GPU:")
    for line in gpu_lines:
        idx, name, total, free = [s.strip() for s in line.split(",")]
        print(f"  GPU {idx}: {name}, 总显存 {int(total)/1024:.1f} GB, 可用 {int(free)/1024:.1f} GB")
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    print(f"未检测到 nvidia-smi: {e}")
    n_gpu = 0

assert n_gpu >= 2, (
    f"本 notebook 需要 >= 2 张 GPU，当前 {n_gpu} 张。"
    "在 Kaggle 上请选择 'GPU T4 x2' 加速器（Settings → Accelerator）。"
)

# 校验：前面没人意外初始化过 CUDA
# （torch 本身可以 import，关键是别调用 torch.cuda.* 触发初始化）
import torch
assert not torch.cuda.is_initialized(), (
    "❌ CUDA 已被初始化！前面某个 cell 调用了 torch.cuda.* 或创建了 GPU tensor。\n"
    "请重启 kernel（Run → Restart Kernel），然后【跳过】之前所有调用 torch.cuda 的 cell，"
    "直接从本 cell 开始重跑。"
)

print(f"\n✅ 双卡环境就绪，且 CUDA 尚未在主进程初始化（fork 安全）")


## Step 2：理解 DDP（数据并行）的工作原理

```
                  单卡（之前的 notebook）              双卡 DDP（本 notebook）
                  ━━━━━━━━━━━━━━━━━━━━━━━           ━━━━━━━━━━━━━━━━━━━━━━━━

  GPU 0:          [完整模型]                          [完整模型副本 #0]
                       ↑                                    ↑
                       │ batch[0..15]                       │ batch[0..7]      ← 半个 batch
                       │                                    │
                       │                              ╔═════╧═════╗
                       │                              ║ NCCL      ║ ← 反向传播时
                       │                              ║ all-reduce║   梯度跨卡同步
                       │                              ╚═════╤═════╝
                       │                                    │
  GPU 1:          [闲置]                              [完整模型副本 #1]
                                                           ↑
                                                           │ batch[8..15]    ← 另一半 batch
```

### DDP 的三条核心规则
1. **每张卡装得下完整模型**（4-bit Qwen2.5-1.5B ≈ 1 GB，T4 16 GB 完全够）
2. **每个 batch 在卡之间均分**（`per_device_train_batch_size=2` × 2 卡 = 全局 batch 4）
3. **反向传播时梯度通过 NCCL all-reduce 同步**（PyTorch DDP 自动做，我们不用管）

### 为什么 device_map="auto" 在这里不能用？
`device_map="auto"` 是**模型并行**——把模型不同层切到不同卡上。它和 DDP（每卡完整副本）是**互斥**的两种范式。
本 notebook 用 **`device_map={"": local_rank}`**，强制每个 rank 把完整模型放到自己那张卡上，符合 DDP 范式。


## Step 3：定义训练函数

`accelerate.notebook_launcher` 的工作方式：
1. 用 `torch.multiprocessing.spawn` 启动 N 个独立 Python 进程
2. 每个进程独立执行你传入的函数
3. 进程之间通过 NCCL 通信（自动配置）

**约束**：训练函数必须**自包含**——所有 import、模型加载、数据准备都要写在函数里面。
原因是 spawn 出来的子进程不会继承 notebook 的 cell 执行状态。

In [ ]:
def training_function():
    """
    DDP 训练入口。每个 rank 独立执行此函数。
    通过 Accelerator().local_process_index 区分自己是 rank 0 还是 rank 1。
    """
    import os, torch
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
        Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    )
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    from accelerate import Accelerator
    from datasets import load_dataset

    MODEL_ID = "Qwen/Qwen2.5-1.5B"
    OUTPUT_DIR = "./qlora_ddp_output"   # 相对路径，Kaggle/Colab/本地都适用
    ADAPTER_DIR = "./qlora_ddp_adapter"

    # ── 0. 当前进程的 rank（0 或 1）────────────────────────────────────
    accelerator = Accelerator()
    local_rank = accelerator.local_process_index
    is_main = accelerator.is_main_process   # 仅 rank 0 为 True

    if is_main:
        print(f"[rank {local_rank}] 总进程数: {accelerator.num_processes}")
        print(f"[rank {local_rank}] 加载 tokenizer 和数据集...")

    # ── 1. Tokenizer ───────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── 2. Alpaca 数据集（取前 1000 条做演示）─────────────────────────
    raw = load_dataset("yahma/alpaca-cleaned", split="train[:1000]")

    PROMPT_TEMPLATE = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n"
        "### Input:\n{input}\n\n"
        "### Response:\n{output}"
    )

    def format_and_tokenize(sample):
        text = PROMPT_TEMPLATE.format(
            instruction=sample["instruction"],
            input=sample["input"] or "",
            output=sample["output"],
        )
        return tokenizer(text, truncation=True, max_length=512, padding=False)

    # 只在主进程打印 map 进度，避免两个 rank 重复刷屏
    with accelerator.main_process_first():
        dataset = raw.map(format_and_tokenize, remove_columns=raw.column_names)

    # ── 3. 加载 4-bit 量化模型到当前 rank 的 GPU ───────────────────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    if is_main:
        print(f"[rank {local_rank}] 加载 {MODEL_ID} 到 cuda:{local_rank}...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        # 关键：DDP 范式下，每个 rank 把完整模型放到自己的 GPU 上
        # 不能用 device_map="auto"（那是模型并行）
        device_map={"": local_rank},
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False    # gradient checkpointing 期间必须关掉

    # ── 4. 应用 LoRA ───────────────────────────────────────────────────
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        # Qwen2.5 是 Llama 风格架构，注意力层叫 q_proj/k_proj/v_proj/o_proj
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    if is_main:
        model.print_trainable_parameters()

    # ── 5. Trainer 配置 ────────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,      # 每张卡 batch=2
        gradient_accumulation_steps=4,       # 梯度累积 4 步
        # 全局有效 batch size = 2 (per device) × 2 (cards) × 4 (accum) = 16
        learning_rate=2e-4,
        fp16=True,                           # T4 用 fp16 混合精度
        logging_steps=5,
        save_strategy="no",
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        # DDP 专属设置：4-bit + LoRA 下推荐关闭这个，否则会有性能开销
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    if is_main:
        n = len(dataset)
        gbs = training_args.per_device_train_batch_size * accelerator.num_processes * training_args.gradient_accumulation_steps
        print(f"[rank {local_rank}] 数据集 {n} 条，全局有效 batch={gbs}，约 {n // gbs} steps/epoch")
        print(f"[rank {local_rank}] 开始训练...")

    trainer.train()

    # ── 6. 只在 rank 0 保存适配器（避免多个进程同时写文件冲突）────────
    if is_main:
        model.save_pretrained(ADAPTER_DIR)
        tokenizer.save_pretrained(ADAPTER_DIR)
        print(f"[rank {local_rank}] 适配器已保存到 {ADAPTER_DIR}")

    # 等所有 rank 跑完再退出
    accelerator.wait_for_everyone()


## Step 4：启动双卡训练

`notebook_launcher` 会做两件事：
1. 用 `torch.distributed.elastic`（默认 fork 启动方式）启动 `num_processes` 个 Python 子进程
2. 在每个子进程里执行 `training_function`

> ⚠️ 再次强调：执行下面这个 cell 时，**主进程必须保持 CUDA 未初始化**。
> 如果你之前不小心跑过 `torch.cuda.*` 调用，需要 **Restart Kernel** 后从 Step 0 重新跑。

整个过程中，notebook 主进程会**阻塞等待**所有子进程完成。
日志会按 rank 0 / rank 1 实时打印（可能交错）。

In [ ]:
from accelerate import notebook_launcher
import torch

# 启动前最后一道保险：确认 CUDA 还没被主进程初始化。
# 这是 notebook_launcher 用 fork 启动子进程的硬性前提。
if torch.cuda.is_initialized():
    raise RuntimeError(
        "CUDA 已经在主进程被初始化了！\n"
        "原因可能是：之前某个 cell 调用了 torch.cuda.is_available() / device_count() / "
        "get_device_properties() 或创建了 GPU tensor。\n"
        "解决方案：菜单 Run → Restart Kernel，然后从 Step 0 重新依次执行；"
        "确保 Step 1 的 GPU 检查只用 nvidia-smi，不调用任何 torch.cuda.* 函数。"
    )

# num_processes=2 → 2 个进程，每个绑定一张 T4
# 注意: notebook_launcher 必须在最外层 cell 调用，不能放在 try/except 等结构里
notebook_launcher(training_function, num_processes=2)


## Step 5：加载训练好的适配器做推理

训练已结束。`notebook_launcher` 的子进程都退出了，主进程的 CUDA 干净如新。
我们重新加载基座 + 训练好的 LoRA 适配器，验证训练效果。

In [ ]:
# 推理只用单卡，避免再次启动 DDP
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B"
ADAPTER_DIR = "./qlora_ddp_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 挂上训练好的 LoRA 适配器
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
model.config.use_cache = True   # 推理打开 KV-cache

PROMPT_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

def generate(instruction, input_text="", max_new_tokens=128):
    prompt = PROMPT_TEMPLATE.format(instruction=instruction, input=input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# 测试
print("=" * 60)
print("Q: What are the three primary colors?")
print("A:", generate("What are the three primary colors?"))
print()
print("=" * 60)
print("Q: Translate to French. Hello, how are you?")
print("A:", generate("Translate the following sentence to French.", "Hello, how are you?"))
print()
print("=" * 60)
print("Q: Write a short poem about the ocean.")
print("A:", generate("Write a short poem about the ocean."))


## Step 6：合并 LoRA 权重，导出可独立部署的模型

和单卡版一样，用 fp16 重新加载基座 → 挂适配器 → merge → save。

In [ ]:
# 重新以 fp16 加载基座（在 CPU 上，避免和 GPU 上的 4-bit 模型抢显存）
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = "./qlora_ddp_merged"

print("以 fp16 重新加载基座（CPU）...")
fp16_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print("挂上训练好的 LoRA 适配器，merge...")
merged = PeftModel.from_pretrained(fp16_base, ADAPTER_DIR).merge_and_unload()

print(f"保存合并后的模型到 {MERGED_DIR}...")
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# 列出文件
print(f"\n{MERGED_DIR}/ 目录:")
total = 0
for f in sorted(os.listdir(MERGED_DIR)):
    sz = os.path.getsize(os.path.join(MERGED_DIR, f))
    total += sz
    print(f"  {f:50s}  {sz/1024/1024:>8.1f} MB")
print(f"  {'合计':50s}  {total/1024/1024/1024:>8.2f} GB")

print("\n之后可以这样加载（不再需要 PEFT）:")
print(f"  model = AutoModelForCausalLM.from_pretrained('{MERGED_DIR}')")


## 总结：单卡 → 双卡 DDP 的关键变更

| 改动点 | 单卡 | 双卡 DDP |
|--------|------|---------|
| 启动方式 | 直接执行 cell | `notebook_launcher(fn, num_processes=2)` |
| 训练逻辑 | 散落在多个 cell | **必须**集中到一个 function 内 |
| 模型放置 | `device_map="auto"` 或不指定 | `device_map={"": local_rank}` |
| GPU 可见性 | `CUDA_VISIBLE_DEVICES=0` | 不限制（要让两张卡都可见） |
| Trainer 参数 | 无 DDP 相关 | `ddp_find_unused_parameters=False` |
| 保存 | 任何时候都可以 save | **只在 rank 0** save，避免文件冲突 |
| 数据预处理 | 直接 `dataset.map(...)` | 包在 `accelerator.main_process_first()` 里 |
| 全局 batch | `per_device * accum` | `per_device * num_gpus * accum` |

## 接下来可以扩展什么？

1. **更大的模型**：本 notebook 1.5B 模型在 T4 上很轻松。可以换成 Qwen2.5-7B（4-bit ~5GB），单卡仍能装下。
2. **更多卡**：把 `num_processes=2` 改成 4 或 8，对应更大的全局 batch size。
3. **真实数据集**：把 `train[:1000]` 改成 `train`（51k 条 Alpaca），训练 3 个 epoch。
4. **更长序列**：`max_length=512` → `max_length=2048`（注意显存会上升 4×）。
5. **bf16 训练**：如果你的 GPU 是 A100/H100，把 `fp16=True` 改成 `bf16=True`，数值更稳定。

## 常见问题

**Q: 看到 `find_unused_parameters` 警告？**
A: PEFT + LoRA 下，部分基座参数确实不参与梯度计算。设 `ddp_find_unused_parameters=False` 提示 DDP 不要扫描，避免性能开销。

**Q: notebook_launcher 卡住不动？**
A: 通常是 NCCL 初始化卡住。Kaggle 上重启 kernel（不是仅 restart cell）通常能解决。

**Q: rank 1 的输出为什么没看到？**
A: 默认只 rank 0 打印训练日志。代码里我用 `if is_main:` 控制了大部分 print。

**Q: 双卡训练完显存还占着？**
A: notebook_launcher 子进程退出后会释放显存。但有时缓存没清，可以 `torch.cuda.empty_cache()`。
